In [1]:
#!/usr/bin/env python3
import os
import re
import numpy as np
import pandas as pd

# === Configuration ===
BASE_PATH = "/home/boat/proxyISP/DGC-Net/proxydgc_eval"  # Update if needed
DIR_PATTERN = re.compile(
    # r"eval_sl_train_v16\.2-chroma-hpatchesv4_pooled480x640_allHomoRepeatedRaw_standardize_gradac8_(\d+)_HpatchesV4"
    r"eval_sl_FIXZEROGRADBUG_CFANORMALIZE_train_v16.2-chroma-HumanTunedInitialHype_sunlit_pooled480x640_allHomoRepeatedRaw_standardize_lr0.0005_gradac32_(\d+)_HpatchesV4"
)
AEPE_FILENAME = "aepe.npy"
NUM_VIEWPOINTS = 5

def load_aepe(filepath):
    """Load AEPE values from .npy or raw file."""
    try:
        return np.load(filepath, allow_pickle=True)
    except Exception:
        with open(filepath, "r") as f:
            content = f.read()
        arr_str = re.findall(r"[\d\.eE\+\-]+", content)
        return np.array([float(x) for x in arr_str])

# === Step 1: Find matching evaluation directories ===
eval_dirs = []
for name in os.listdir(BASE_PATH):
    full_path = os.path.join(BASE_PATH, name)
    if os.path.isdir(full_path):
        match = DIR_PATTERN.match(name)
        if match:
            iter_num = int(match.group(1))
            eval_dirs.append((iter_num, full_path))

if not eval_dirs:
    print("❌ No matching eval directories found.")
    
eval_dirs.sort(key=lambda x: x[0])

# === Step 2: Gather AEPE values ===
rows = []
for iter_num, dir_path in eval_dirs:
    aepe_path = os.path.join(dir_path, AEPE_FILENAME)
    if os.path.exists(aepe_path):
        data = load_aepe(aepe_path)
        if len(data) == NUM_VIEWPOINTS:
            rows.append({
                "iter": iter_num,
                "dir_name": os.path.basename(dir_path)[-20:],
                "avg_aepe": np.mean(data),
                **{f"v{i+1}": data[i] for i in range(NUM_VIEWPOINTS)}
            })
        else:
            print(f"[WARN] Skipping {dir_path}: expected {NUM_VIEWPOINTS} values, got {len(data)}")
    else:
        print(f"[WARN] {AEPE_FILENAME} not found in {dir_path}")

if not rows:
    print("❌ No valid AEPE data found.")

# === Step 3: Create and sort table ===
df = pd.DataFrame(rows)
df_sorted = df.sort_values(by="avg_aepe", ascending=True)

# === Step 4: Display results ===
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)  # Show all rows
print("\n🟢 Sorted AEPE results (ascending avg_aepe):")
df_sorted.to_string(index=False)
df_sorted




🟢 Sorted AEPE results (ascending avg_aepe):


,iter,dir_name,avg_aepe,v1,v2,v3,v4,v5
9,105000,32_105000_HpatchesV4,5.473507,0.694477,0.761725,1.644409,6.022231,18.244692
47,143000,32_143000_HpatchesV4,5.486964,0.686775,0.754051,1.647717,6.023034,18.323242
8,104000,32_104000_HpatchesV4,5.488271,0.695885,0.761995,1.636009,6.027987,18.319478
5,101000,32_101000_HpatchesV4,5.488712,0.693653,0.758222,1.638458,6.012646,18.340583
19,115000,32_115000_HpatchesV4,5.489439,0.694597,0.761874,1.657399,6.000001,18.333324
39,135000,32_135000_HpatchesV4,5.491561,0.689298,0.754584,1.630984,6.017722,18.365220
33,129000,32_129000_HpatchesV4,5.491890,0.693262,0.758979,1.651298,6.054797,18.301116
22,118000,32_118000_HpatchesV4,5.492526,0.690715,0.756771,1.641574,5.980517,18.393052
28,124000,32_124000_HpatchesV4,5.493117,0.691503,0.757299,1.651589,6.023894,18.341298
4,100000,32_100000_HpatchesV4,5.493785,0.694210,0.763628,1.645114,6.049881,18.316093
